# 2hClub — ball detector training on outside footage (Colab)

Trains the ball detector on `tiles.zip` from `vision/build_external.py`: tiles made only from two
CC BY 4.0 Roboflow Universe datasets (other matches). No frame of the provided match is used.

1. Runtime → Change runtime type → **T4 GPU** (cheapest per hour; training takes about 1.5–2 h).
2. Upload `tiles.zip` anywhere in your Google Drive (the notebook finds it).
3. Runtime → **Run all**. Progress is saved to `MyDrive/2hclub/runs/<RUN_NAME>/` after every epoch.
   The last cell disconnects the GPU by itself, so no units are spent after training.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Settings. Our ball is ~6 px in a 640 tile, so we train (and later detect) at 1280: ~12 px.
ZIP_NAME = 'tiles.zip'
MODEL = 'yolo26s.pt'      # COCO-pretrained (Ultralytics, public)
IMGSZ = 1280
EPOCHS = 25
BATCH = 16                # the trainer lowers it by itself if the GPU is too small
RUN_NAME = 'ball_ext1'
SEED = 0
RUNS_DIR = '/content/drive/MyDrive/2hclub/runs'
MIN_PRECISION = 0.90      # the confidence threshold is the lowest one reaching this on the test tiles

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install ultralytics==8.4.190

In [ ]:
import glob, os, zipfile
found = glob.glob(f'/content/drive/MyDrive/**/{ZIP_NAME}', recursive=True)
assert found, f'{ZIP_NAME} not found anywhere in your Google Drive: upload it first'
print('using', found[0])
with zipfile.ZipFile(found[0]) as z:
    z.extractall('/content/tiles')
with open('/content/tiles/data.yaml', 'w') as f:
    f.write('path: /content/tiles\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n  0: ball\n')
for split in ('train', 'val', 'test'):
    labels = glob.glob(f'/content/tiles/labels/{split}/*.txt')
    print(split, len(labels), 'tiles,', sum(os.path.getsize(p) > 0 for p in labels), 'with a ball')

In [ ]:
from ultralytics import YOLO
model = YOLO(MODEL)
model.train(
    data='/content/tiles/data.yaml', imgsz=IMGSZ, epochs=EPOCHS, batch=BATCH,
    seed=SEED, deterministic=True, patience=10, cos_lr=True, cache='ram',
    # Tiny objects: little zoom (the tiles already vary the ball from 4 to 9 px), flips and colour.
    scale=0.2, mosaic=1.0, close_mosaic=5, fliplr=0.5, degrees=0.0,
    project=RUNS_DIR, name=RUN_NAME, exist_ok=True, plots=True, save_period=1,
)

In [ ]:
# Choose the confidence threshold on the held-out test tiles (outside footage only).
import json, numpy as np
best = f'{RUNS_DIR}/{RUN_NAME}/weights/best.pt'
model = YOLO(best)
truth, preds = {}, []
for lab in sorted(glob.glob('/content/tiles/labels/test/*.txt')):
    stem = os.path.basename(lab)[:-4]
    rows = [l.split() for l in open(lab) if l.strip()]
    truth[stem] = [(float(r[1]) * 640, float(r[2]) * 640) for r in rows]
for r in model.predict('/content/tiles/images/test', imgsz=IMGSZ, conf=0.01, stream=True, verbose=False):
    stem = os.path.basename(r.path)[:-4]
    for (x1, y1, x2, y2), c in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy()):
        preds.append((stem, (x1 + x2) / 2, (y1 + y2) / 2, float(c)))
n_true = sum(len(v) for v in truth.values())
rows = []
for t in np.arange(0.05, 0.96, 0.05):
    used, tp, fp = set(), 0, 0
    for stem, x, y, c in sorted([p for p in preds if p[3] >= t], key=lambda p: -p[3]):
        hit = [i for i, (gx, gy) in enumerate(truth[stem]) if (stem, i) not in used and np.hypot(x - gx, y - gy) <= 6]
        if hit:
            used.add((stem, hit[0])); tp += 1
        else:
            fp += 1
    p = tp / (tp + fp) if tp + fp else 1.0
    rows.append((round(float(t), 2), p, tp / n_true))
    print(f'conf {t:.2f}  precision {p:.3f}  recall {tp / n_true:.3f}')
ok = [r for r in rows if r[1] >= MIN_PRECISION and r[2] > 0]
conf = ok[0][0] if ok else rows[-1][0]
out = {'conf': conf, 'min_precision': MIN_PRECISION, 'curve': rows}
with open(f'{RUNS_DIR}/{RUN_NAME}/conf.json', 'w') as f:
    json.dump(out, f, indent=1)
print('chosen confidence threshold', conf, '-> saved conf.json next to the weights')

In [ ]:
# Done: free the GPU so no more units are used. (Results are already in Drive.)
from google.colab import runtime
runtime.unassign()